# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-4o-mini"
print("client ready:", MODEL)

client ready: gpt-4o-mini


## 1) Task 1 — 단발 호출 + 토큰/비용 확인

In [7]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
        {"role": "user", "content": "사내 챗봇을 한 문장으로 소개해줘."},
    ],
    temperature=0.9,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

사내 챗봇은 직원들이 빠르고 효율적으로 정보를 얻고 소통할 수 있도록 돕는 인공지능 기반 도구입니다.
usage: CompletionUsage(completion_tokens=33, prompt_tokens=39, total_tokens=72, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


### 스트리밍 — 토큰이 생성되는 대로 출력

In [3]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "LLM 을 3줄로 설명해줘."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="")
print()

LLM(대형 언어 모델)은 방대한 양의 텍스트 데이터를 기반으로 학습하여 언어 이해와 생성 능력을 가진 인공지능 모델입니다. 자연어 처리(NLP) 작업에서 질문 응답, 번역, 요약 등 다양한 용도로 사용됩니다. 이러한 모델은 텍스트의 의미와 맥락을 파악하여 인간과 비슷한 방식으로 언어를 생성할 수 있습니다.


## 2) Task 2 — 리뷰 100개 긍/부정 분류 (LLM)

실습 1의 산출물 `reviews_clean.parquet` 에서 100건을 샘플링해 분류한다.
- `temperature=0` (분류는 결정적이어야 함)
- **JSON 응답 강제** (`response_format`) — 파싱 안전

In [8]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]                      # 중립 제외 (실습 2 와 동일 기준)
df["label"] = (df["rating"] >= 4).astype(int)   # 정답: 4~5 긍정(1), 1~2 부정(0)
sample = df.sample(100, random_state=42).reset_index(drop=True)
print(sample["label"].value_counts())
sample[["clean_text", "label"]].head()

label
1    54
0    46
Name: count, dtype: int64


,clean_text,label
0,사진과 색이 완전히 달라서 실망했어요,0
1,한 번 쓰고 고장났습니다 환불 원해요,0
2,포장도 꼼꼼하고 품질이 기대 이상이에요,1
3,색감이 사진이랑 똑같아서 만족합니다,1
4,냄새가 너무 심해서 못 쓰겠어요,0


In [10]:
import json

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    'JSON 으로만 답한다: {"label": 0 또는 1}'
)

def classify(text: str) -> tuple[int, int]:   # (라벨, 사용 토큰) 을 함께 반환
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},   # JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    return int(data["label"]), resp.usage.total_tokens   # 라벨 + 비용계산용 토큰

# 1건 점검 — (라벨, 토큰) 튜플이 찍힌다
print(sample.loc[0, "clean_text"])
label, used = classify(sample.loc[0, "clean_text"])
print("라벨:", label, "/ 토큰:", used)

사진과 색이 완전히 달라서 실망했어요
라벨: 0 / 토큰: 82


In [11]:
preds, tokens = [], 0
for t in sample["clean_text"]:
    label, used = classify(t)
    preds.append(label)
    tokens += used
sample["pred"] = preds
print("총 토큰:", tokens)

총 토큰: 8117


### sklearn(실습 2) 과 비교 — 정확도

In [20]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["label"], sample["pred"])
print(f"LLM 정확도: {acc:.3f}")
print(classification_report(sample["label"], sample["pred"], digits=3))
# TODO: 실습 2 sklearn 정확도와 한 표로 비교 (정확도 / 비용 / 속도 / 일관성)

LLM 정확도: 0.990
              precision    recall  f1-score   support

           0      1.000     0.978     0.989        46
           1      0.982     1.000     0.991        54

    accuracy                          0.990       100
   macro avg      0.991     0.989     0.990       100
weighted avg      0.990     0.990     0.990       100



## 3) Task 3 — 비용 측정 + 1만 건 추산

In [22]:
# gpt-4o-mini 단가 (2026 기준, 변동 가능) — 슬라이드 '비용·운영 한눈에' 참고
# 입력 0.15/M, 출력 0.60/M $. 분류는 출력이 5토큰 내외로 매우 짧아 출력 비용은 무시하고
# total_tokens 를 입력 단가로 보수적으로 추정한다.
PRICE_IN = 0.15 / 1_000_000   # 입력 토큰당 $

avg_tokens = tokens / len(sample)
cost_100   = tokens * PRICE_IN                 # 보수적으로 입력 단가 적용
print(f"평균 토큰/건: {avg_tokens:.1f}")
print(f"100건 비용: ${cost_100:.4f}")
print(f"1만 건 추산: ${cost_100 * 100:.2f}")
# TODO: ML vs LLM — '언제 무엇을 쓸지' 표로 구현해준다

평균 토큰/건: 81.2
100건 비용: $0.0012
1만 건 추산: $0.12


## 회고 / 산출물
- [ ] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [ ] 한 줄 결론 — *대량·단순 분류는 ___, 유연·복잡 추론은 ___*

In [26]:
import json

반어_리뷰 = [
    "어머니가 좋아하시겠네요, 1980년대 물건같아요",
    "품질 최고예요~ 환불하고 싶을 만큼",
    "빠른 배송 감사합니다 일주일이나 걸려서요",
]

SYSTEM_R = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. 입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    "반드시 JSON으로만 응답한다: {\"label\": 0 또는 1, \"confidence\": 0.0-1.0, \"reason\": \"간단한 한 문장 사유\"}\n"
    "reason은 모델이 해당 라벨을 선택한 간단한 이유(한국어, 1문장 이내)를 제공한다."
)

def classify_with_reason(text: str):
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_R},
            {"role": "user", "content": text},
        ],
    )
    content = resp.choices[0].message.content
    try:
        data = json.loads(content)
    except Exception:
        # JSON 파싱 실패 시 본문에서 JSON 객체 추출 시도
        import re
        m = re.search(r"\{.*\}", content, re.S)
        if m:
            try:
                data = json.loads(m.group())
            except Exception:
                data = {"label": 0, "confidence": 0.0, "reason": content.strip()}
        else:
            data = {"label": 0, "confidence": 0.0, "reason": content.strip()}
    label = int(data.get("label", 0))
    confidence = float(data.get("confidence", 0.0))
    reason = data.get("reason", "")
    return label, confidence, reason

for r in 반어_리뷰:
    label, conf, reason = classify_with_reason(r)
    lbl = '긍정' if label == 1 else '부정'
    print(f"리뷰: {r}\n-> 라벨: {lbl} ({label}), 확신도: {conf:.2f}, 사유: {reason}\n")


리뷰: 어머니가 좋아하시겠네요, 1980년대 물건같아요
-> 라벨: 긍정 (1), 확신도: 0.85, 사유: 어머니가 좋아하실 것이라는 긍정적인 표현이 포함되어 있습니다.

리뷰: 품질 최고예요~ 환불하고 싶을 만큼
-> 라벨: 부정 (0), 확신도: 0.85, 사유: 환불하고 싶을 만큼이라는 표현이 부정적인 감정을 나타냅니다.

리뷰: 빠른 배송 감사합니다 일주일이나 걸려서요
-> 라벨: 부정 (0), 확신도: 0.85, 사유: 배송이 빠르지 않았다는 부정적인 언급이 있다.

